# Structured Data Loader — Neo4j AuraDB

Loads all entity nodes and relationship edges from `data/layer_1/entities/` and `data/layer_1/links/` into Neo4j AuraDB.

| Step | What happens |
|------|--------------|
| 1 | Setup & env |
| 2 | Connect to AuraDB |
| 3 | Clear database |
| 4 | Create constraints & indexes |
| 5 | Load entity nodes |
| 6 | Load relationship edges |
| 7 | Verify row counts |
| 8 | Close connection |

> All `MERGE` statements are idempotent — safe to re-run.

## 1. Setup

In [1]:
import sys, os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")

ENTITIES_DIR = project_root / "data" / "layer_1" / "entities"
LINKS_DIR    = project_root / "data" / "layer_1" / "links"

for var in ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"):
    status = "set" if os.getenv(var) else "MISSING"
    print(f"  {var}: {status}")

  NEO4J_URI: set
  NEO4J_USERNAME: set
  NEO4J_PASSWORD: set


## 2. Connect to AuraDB

In [ ]:
from src.graph.connection import Neo4jConnection

conn = Neo4jConnection()
conn.connect()

result = conn.run_query("RETURN 'Connected to AuraDB' AS status")
print(result[0]["status"])

## 3. Clear Database

Wipes all nodes and relationships before loading. Safe to skip on incremental re-runs since all `MERGE` statements are idempotent.

In [ ]:
result = conn.run_query("MATCH (n) DETACH DELETE n")
print("Database cleared.")

## 4. Constraints & Indexes

Unique constraints prevent duplicate nodes and speed up `MERGE` lookups.

In [ ]:
CONSTRAINTS = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Borrower)        REQUIRE n.borrower_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:LoanApplication) REQUIRE n.loan_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:BankAccount)     REQUIRE n.account_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Transaction)     REQUIRE n.transaction_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Address)         REQUIRE n.address_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Collateral)      REQUIRE n.collateral_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Officer)         REQUIRE n.officer_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Jurisdiction)    REQUIRE n.jurisdiction_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (n:Industry)        REQUIRE n.industry_id IS UNIQUE",
]

for stmt in CONSTRAINTS:
    conn.run_query(stmt)

print(f"Applied {len(CONSTRAINTS)} constraints.")

## 5. Load Entity Nodes & Relationships — Helper Functions

`load_nodes` optionally promotes a column value as a secondary label via `extra_label_field`
(e.g. `Borrower:Individual`, `Borrower:Corporate`).
Both functions batch via `UNWIND … MERGE` and are idempotent.

In [ ]:
import re

def _to_label(value: str) -> str:
    """Convert snake_case value to PascalCase Neo4j label.
    e.g. 'personal_transaction' → 'PersonalTransaction'
    """
    return "".join(word.capitalize() for word in re.split(r"[_\s]+", str(value)))


def load_nodes(
    conn,
    df: pd.DataFrame,
    label: str,
    id_field: str,
    extra_label_field: str = None,
    batch_size: int = 500,
) -> int:
    """MERGE nodes in batches.

    If extra_label_field is given, rows are grouped by that column and merged
    with a compound label (e.g. Borrower:Individual). The promoted column is
    dropped from node properties since it is encoded in the label.
    Returns total rows written.
    """
    total = 0
    groups = df.groupby(extra_label_field) if extra_label_field else [("", df)]
    for extra_val, group in groups:
        label_expr = f"{label}:{_to_label(extra_val)}" if extra_label_field else label
        props = group.drop(columns=[extra_label_field]) if extra_label_field else group
        records = props.where(pd.notnull(props), None).to_dict("records")
        cypher = (
            f"UNWIND $rows AS row "
            f"MERGE (n:{label_expr} {{{id_field}: row.{id_field}}}) "
            f"SET n += row"
        )
        for i in range(0, len(records), batch_size):
            conn.run_query(cypher, {"rows": records[i : i + batch_size]})
            total += len(records[i : i + batch_size])
    return total


def load_rels(conn, df: pd.DataFrame, cypher: str, batch_size: int = 500) -> int:
    """MERGE relationships in batches. Returns total rows written."""
    records = df.where(pd.notnull(df), None).to_dict("records")
    total = 0
    for i in range(0, len(records), batch_size):
        conn.run_query(cypher, {"rows": records[i : i + batch_size]})
        total += len(records[i : i + batch_size])
    return total


print("Helper functions defined.")

In [ ]:
# Borrowers → :Borrower:Individual or :Borrower:Corporate
df = pd.read_csv(ENTITIES_DIR / "borrowers.csv")
n = load_nodes(conn, df, "Borrower", "borrower_id", extra_label_field="type")
print(f"Borrower        : {n} nodes")

# Loan Applications → :LoanApplication:ResidentialSecured or :LoanApplication:CommercialSecured
df = pd.read_csv(ENTITIES_DIR / "loan_applications.csv")
n = load_nodes(conn, df, "LoanApplication", "loan_id", extra_label_field="loan_type")
print(f"LoanApplication : {n} nodes")

# Bank Accounts → :BankAccount:PersonalTransaction | :BusinessTransaction | :CorporateCurrent
df = pd.read_csv(ENTITIES_DIR / "bank_accounts.csv")
n = load_nodes(conn, df, "BankAccount", "account_id", extra_label_field="account_type")
print(f"BankAccount     : {n} nodes")

# Transactions — FK columns dropped; account links expressed via FROM_ACCOUNT / TO_ACCOUNT
df = pd.read_csv(ENTITIES_DIR / "transactions.csv").drop(columns=["from_account_id", "to_account_id"])
df["flagged_suspicious"] = df["flagged_suspicious"].astype(str).str.lower().map({"true": True, "false": False})
n = load_nodes(conn, df, "Transaction", "transaction_id")
print(f"Transaction     : {n} nodes")

# Addresses → :Address:Residential or :Address:Commercial
df = pd.read_csv(ENTITIES_DIR / "addresses.csv")
n = load_nodes(conn, df, "Address", "address_id", extra_label_field="address_type")
print(f"Address         : {n} nodes")

# Collateral
df = pd.read_csv(ENTITIES_DIR / "collateral.csv")
df["encumbered"] = df["encumbered"].astype(str).str.lower().map({"true": True, "false": False})
n = load_nodes(conn, df, "Collateral", "collateral_id")
print(f"Collateral      : {n} nodes")

# Officers
df = pd.read_csv(ENTITIES_DIR / "officers.csv")
for col in ("is_pep", "sanctions_match"):
    df[col] = df[col].astype(str).str.lower().map({"true": True, "false": False})
n = load_nodes(conn, df, "Officer", "officer_id")
print(f"Officer         : {n} nodes")

# Jurisdictions → :Jurisdiction:Federal | :National | :State | :SpecialAdminRegion
df = pd.read_csv(ENTITIES_DIR / "jurisdictions.csv")
n = load_nodes(conn, df, "Jurisdiction", "jurisdiction_id", extra_label_field="level")
print(f"Jurisdiction    : {n} nodes")

# Industries
df = pd.read_csv(ENTITIES_DIR / "industries.csv")
n = load_nodes(conn, df, "Industry", "industry_id")
print(f"Industry        : {n} nodes")

## 6. Load Relationship Edges

In [ ]:
# (LoanApplication)-[:SUBMITTED_BY {role}]->(Borrower)
df = pd.read_csv(LINKS_DIR / "submitted_by.csv")
cypher = """
UNWIND $rows AS row
MATCH (l:LoanApplication {loan_id: row.loan_id})
MATCH (b:Borrower        {borrower_id: row.borrower_id})
MERGE (l)-[r:SUBMITTED_BY]->(b)
SET r.role = row.role
"""
n = load_rels(conn, df, cypher)
print(f"SUBMITTED_BY         : {n} rels")


# (Borrower|LoanApplication)-[:HAS_ACCOUNT {role, authorized_signatory}]->(BankAccount)
df = pd.read_csv(LINKS_DIR / "has_account.csv")
df["authorized_signatory"] = df["authorized_signatory"].astype(str).str.lower().map({"true": True, "false": False})
for etype, label, id_col in [("borrower", "Borrower", "borrower_id"), ("loan_application", "LoanApplication", "loan_id")]:
    sub = df[df["entity_type"] == etype].copy()
    sub = sub.rename(columns={"entity_id": id_col})
    if sub.empty:
        continue
    cypher = f"""
    UNWIND $rows AS row
    MATCH (e:{label} {{{id_col}: row.{id_col}}})
    MATCH (a:BankAccount {{account_id: row.account_id}})
    MERGE (e)-[r:HAS_ACCOUNT]->(a)
    SET r.role = row.role, r.authorized_signatory = row.authorized_signatory
    """
    n = load_rels(conn, sub, cypher)
    print(f"HAS_ACCOUNT ({etype:20s}): {n} rels")


# (Borrower)-[:LOCATED_AT {address_use, is_current, effective_date}]->(Address)
df = pd.read_csv(LINKS_DIR / "located_at.csv")
df["is_current"] = df["is_current"].astype(str).str.lower().map({"true": True, "false": False})
for etype, label, id_col in [("borrower", "Borrower", "borrower_id")]:
    sub = df[df["entity_type"] == etype].copy()
    sub = sub.rename(columns={"entity_id": id_col})
    cypher = f"""
    UNWIND $rows AS row
    MATCH (e:{label} {{{id_col}: row.{id_col}}})
    MATCH (a:Address {{address_id: row.address_id}})
    MERGE (e)-[r:LOCATED_AT]->(a)
    SET r.address_use = row.address_use, r.is_current = row.is_current, r.effective_date = row.effective_date
    """
    n = load_rels(conn, sub, cypher)
    print(f"LOCATED_AT           : {n} rels")


# (LoanApplication)-[:BACKED_BY {lien_position, coverage_ratio}]->(Collateral)
df = pd.read_csv(LINKS_DIR / "backed_by.csv")
cypher = """
UNWIND $rows AS row
MATCH (l:LoanApplication {loan_id:       row.loan_id})
MATCH (c:Collateral      {collateral_id: row.collateral_id})
MERGE (l)-[r:BACKED_BY]->(c)
SET r.lien_position = row.lien_position, r.coverage_ratio = row.coverage_ratio
"""
n = load_rels(conn, df, cypher)
print(f"BACKED_BY            : {n} rels")


# (LoanApplication)-[:GUARANTEED_BY {guarantee_type, guarantee_amount, currency}]->(Borrower)
df = pd.read_csv(LINKS_DIR / "guaranteed_by.csv")
cypher = """
UNWIND $rows AS row
MATCH (l:LoanApplication {loan_id:     row.loan_id})
MATCH (b:Borrower        {borrower_id: row.borrower_id})
MERGE (l)-[r:GUARANTEED_BY]->(b)
SET r.guarantee_type = row.guarantee_type, r.guarantee_amount = row.guarantee_amount, r.currency = row.currency
"""
n = load_rels(conn, df, cypher)
print(f"GUARANTEED_BY        : {n} rels")


# (Officer)-[:DIRECTOR_OF {role, appointed_date, is_current}]->(Borrower)
df = pd.read_csv(LINKS_DIR / "director_of.csv")
df["is_current"] = df["is_current"].astype(str).str.lower().map({"true": True, "false": False})
cypher = """
UNWIND $rows AS row
MATCH (o:Officer  {officer_id:  row.officer_id})
MATCH (b:Borrower {borrower_id: row.borrower_id})
MERGE (o)-[r:DIRECTOR_OF]->(b)
SET r.role = row.role, r.appointed_date = row.appointed_date, r.is_current = row.is_current
"""
n = load_rels(conn, df, cypher)
print(f"DIRECTOR_OF          : {n} rels")


# (Borrower)-[:BELONGS_TO_INDUSTRY {is_primary, revenue_percentage}]->(Industry)
df = pd.read_csv(LINKS_DIR / "belongs_to_industry.csv")
df["is_primary"] = df["is_primary"].astype(str).str.lower().map({"true": True, "false": False})
cypher = """
UNWIND $rows AS row
MATCH (b:Borrower {borrower_id: row.borrower_id})
MATCH (i:Industry {industry_id: row.industry_id})
MERGE (b)-[r:BELONGS_TO_INDUSTRY]->(i)
SET r.is_primary = row.is_primary, r.revenue_percentage = row.revenue_percentage
"""
n = load_rels(conn, df, cypher)
print(f"BELONGS_TO_INDUSTRY  : {n} rels")


# (Borrower:Corporate)-[:REGISTERED_IN {registration_type, registration_number, registration_date}]->(Jurisdiction)
# Only corporate borrowers — sourced from registered_in.csv
df = pd.read_csv(LINKS_DIR / "registered_in.csv")
cypher = """
UNWIND $rows AS row
MATCH (b:Borrower     {borrower_id:     row.borrower_id})
MATCH (j:Jurisdiction {jurisdiction_id: row.jurisdiction_id})
MERGE (b)-[r:REGISTERED_IN]->(j)
SET r.registration_type   = row.registration_type,
    r.registration_number = row.registration_number,
    r.registration_date   = row.registration_date
"""
n = load_rels(conn, df, cypher)
print(f"REGISTERED_IN        : {n} rels  (corporate only)")


# (Borrower:Individual)-[:RESIDES_IN {residency_type, tax_id}]->(Jurisdiction)
# Individual borrowers (Australian tax residents) — sourced from resides_in.csv
df = pd.read_csv(LINKS_DIR / "resides_in.csv")
cypher = """
UNWIND $rows AS row
MATCH (b:Borrower     {borrower_id:     row.borrower_id})
MATCH (j:Jurisdiction {jurisdiction_id: row.jurisdiction_id})
MERGE (b)-[r:RESIDES_IN]->(j)
SET r.residency_type = row.residency_type,
    r.tax_id         = row.tax_id
"""
n = load_rels(conn, df, cypher)
print(f"RESIDES_IN           : {n} rels  (individual only)")


# (Borrower)-[:OWNS {ownership_percentage, ownership_type, effective_date}]->(Borrower)
df = pd.read_csv(LINKS_DIR / "owns.csv")
cypher = """
UNWIND $rows AS row
MATCH (owner:Borrower {borrower_id: row.owner_id})
MATCH (owned:Borrower {borrower_id: row.owned_id})
MERGE (owner)-[r:OWNS]->(owned)
SET r.ownership_percentage = row.ownership_percentage,
    r.ownership_type       = row.ownership_type,
    r.effective_date       = row.effective_date
"""
n = load_rels(conn, df, cypher)
print(f"OWNS                 : {n} rels")


# Transactions: (BankAccount)-[:FROM_ACCOUNT]->(Transaction)-[:TO_ACCOUNT]->(BankAccount)
df = pd.read_csv(ENTITIES_DIR / "transactions.csv")[["transaction_id", "from_account_id", "to_account_id"]]
cypher = """
UNWIND $rows AS row
MATCH (t:Transaction {transaction_id: row.transaction_id})
MATCH (src:BankAccount {account_id: row.from_account_id})
MATCH (dst:BankAccount {account_id: row.to_account_id})
MERGE (src)-[:FROM_ACCOUNT]->(t)
MERGE (t)-[:TO_ACCOUNT]->(dst)
"""
n = load_rels(conn, df, cypher)
print(f"FROM_ACCOUNT / TO_ACCOUNT : {n} transaction stubs")


## 7. Verify Node & Relationship Counts

In [ ]:
labels = [
    "Borrower", "LoanApplication", "BankAccount", "Transaction",
    "Address", "Collateral", "Officer", "Jurisdiction", "Industry"
]
print("Node counts:")
for label in labels:
    result = conn.run_query(f"MATCH (n:{label}) RETURN count(n) AS cnt")
    print(f"  {label:20s} : {result[0]['cnt']}")

print()
rel_types = [
    "SUBMITTED_BY", "HAS_ACCOUNT", "LOCATED_AT", "BACKED_BY",
    "GUARANTEED_BY", "DIRECTOR_OF", "BELONGS_TO_INDUSTRY",
    "REGISTERED_IN", "RESIDES_IN",
    "OWNS", "FROM_ACCOUNT", "TO_ACCOUNT",
]
print("Relationship counts:")
for rel in rel_types:
    result = conn.run_query(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS cnt")
    print(f"  {rel:20s} : {result[0]['cnt']}")


## 8. Close Connection

In [ ]:
conn.close()
print("Connection closed.")